## 定义模型

In [34]:
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv
load_dotenv()

provider = "dashscope"  # 可选 "modelscope", "dashscope" "openrouter"
model = "qwen-flash"

if provider == "modelscope":
    model = model if model else os.getenv("MODELSCOPE_MODEL")
    base_url = os.getenv("MODELSCOPE_BASE_URL")
    api_key = os.getenv("MODELSCOPE_API_KEY")
elif provider == "dashscope":
    model = model if model else os.getenv("DASHSCOPE_MODEL")
    base_url = os.getenv("DASHSCOPE_BASE_URL")
    api_key = os.getenv("DASHSCOPE_API_KEY")
elif provider == "openrouter":
    model = model if model else os.getenv("OPENROUTER_MODEL")
    base_url = os.getenv("OPENROUTER_BASE_URL")
    api_key = os.getenv("OPENROUTER_API_KEY")
print(f"Using model [{model}] from provider [{provider}]")

llm = init_chat_model(
    model=model,
    model_provider="openai",
    base_url=base_url,
    api_key=api_key,
)

Using model [qwen-flash] from provider [dashscope]


## 定义结构化输出


In [35]:
# Schema for structured output
from pydantic import BaseModel, Field


class SearchQuery(BaseModel):
    search_query: str = Field(None, description="优化后的搜索查询")
    justification: str = Field(
        None, description="为什么这个查询与用户的请求相关，简短说明。"
    )

## 给模型绑定结构化输出

In [36]:
# Augment the LLM with schema for structured output
structured_llm = llm.with_structured_output(SearchQuery)

# Invoke the augmented LLM
output = structured_llm.invoke("如何做一个蛋挞？")

print(f"查询：{output.search_query}")
print(f"理由：{output.justification}")

查询：如何在家做蛋挞
理由：为了提供一个清晰、准确且易于操作的蛋挞制作方法，我将根据常见的家庭做法整理步骤，并确保材料和工具简单易得。


/opt/miniconda3/envs/onto/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=SearchQuery(search_query=...工具简单易得。'), input_type=SearchQuery])
  return self.__pydantic_serializer__.to_python(


## 模型提供商专属接口
上面OpenAI兼容模式不能保证所有所有模型厂商稳定输出，为保证稳定性，可以使用langchain生态的特定接口

In [37]:
from langchain_community.chat_models import ChatTongyi
llm = ChatTongyi(
    model="qwen3-235b-a22b-instruct-2507",  # 推荐使用 qwen-max 或 qwen-plus 以获得更好的指令遵循能力
    dashscope_api_key=os.getenv("DASHSCOPE_API_KEY"),
    temperature=0.0    # 结构化输出建议将温度设为 0
)
structured_llm = llm.with_structured_output(SearchQuery)
output = structured_llm.invoke("如何做一个蛋挞？")

print(f"查询：{output.search_query}")
print(f"理由：{output.justification}")

查询：蛋挞制作方法 步骤 家庭做法
理由：用户想知道如何制作蛋挞，需要获取详细的制作步骤和所需材料。
